# 🛩️ IHDP Agent: Failure Injection Demo

**Environment:** `LinearLongitudinalF16-v0` | **Control:** Incremental Heuristic Dynamic Programming

---

This notebook demonstrates how the **IHDP (Incremental Heuristic Dynamic Programming)** controller adapts to sudden changes in aircraft dynamics — simulating an in-flight failure scenario on the **F-16 longitudinal model**.

**Paper-equation update:** the SISO actor gradient now includes the physical output scale. The actor learning rate and its floor are expressed in these units. Previous cached results were cleared; execute this notebook to obtain results with the current implementation.

## 📋 What You'll Learn

| Step | Description |
|------|-------------|
| **1** | Set up a step-tracking task for angle of attack (α) |
| **2** | Configure and run an IHDP controller for one episode |
| **3** | Inject a "failure" mid-flight by modifying the A-matrix |
| **4** | Visualize how the controller adapts to changed dynamics |

> 💡 **Key Insight:** IHDP is an adaptive neural-network-based controller that can adjust to model uncertainties and failures in real-time.

## 📑 Table of Contents

1. [Prerequisites](#prerequisites)
2. [Imports & Setup](#1-imports--setup)
3. [Experiment Configuration](#2-experiment-configuration)
4. [Environment Setup](#3-environment-setup)
5. [IHDP Agent Configuration](#4-ihdp-agent-configuration)
6. [Simulation with Failure Injection](#5-simulation-with-failure-injection)
7. [Results Visualization](#6-results-visualization)
8. [Summary](#summary)

<a id="prerequisites"></a>
## ⚙️ Prerequisites

Before running this notebook, ensure you have:

```bash
# Option 1: Install via poetry (recommended for development)
poetry install --with jupyter

# Option 2: Install via pip
pip install tensoraerospace[benchmark]
```

**Requirements:**
- Python 3.10+
- `tensoraerospace` package installed
- `gymnasium` (included with tensoraerospace)
- JupyterLab / Notebook / VS Code with Python extension

---

<a id="1-imports--setup"></a>
## 1️⃣ Imports & Setup

All imports are consolidated here for easy copy-paste into other projects.

In [ ]:
# Suppress all warnings for cleaner output
import warnings
import os
warnings.filterwarnings('ignore')

# Core dependencies
import gymnasium as gym
import numpy as np
import pandas as pd
from tqdm import tqdm

# TensorAeroSpace components
import tensoraerospace  # Registers Gymnasium environments
from tensoraerospace.agent.ihdp.model import IHDPAgent
from tensoraerospace.signals.standard import unit_step
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

print("✅ All imports successful!")

---

<a id="2-experiment-configuration"></a>
## 2️⃣ Experiment Configuration

All tunable parameters are collected in one place for easy modification.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#                           EXPERIMENT CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

CONFIG = {
    # Simulation parameters
    "dt": 0.01,              # Time step [seconds]
    "tn": 40,                # Total simulation time [seconds]
    
    # Reference signal (step input)
    "step_amplitude_deg": 5, # Step amplitude in degrees
    "step_time": 10,         # Time when step occurs [seconds]
    
    # Failure injection
    "failure_step": 2500,    # Step number to inject failure (25 seconds)
    "failure_value": 0.98,   # Modified A-matrix element value
    
    # Tracking state
    "tracking_state": "alpha",  # Angle of attack
}

# Display configuration as a clean table
config_df = pd.DataFrame([
    ("Time step (dt)", f"{CONFIG['dt']} s"),
    ("Simulation duration", f"{CONFIG['tn']} s"),
    ("Step amplitude", f"{CONFIG['step_amplitude_deg']}°"),
    ("Step time", f"{CONFIG['step_time']} s"),
    ("Failure injection at", f"step {CONFIG['failure_step']} ({CONFIG['failure_step'] * CONFIG['dt']:.1f} s)"),
    ("Tracking state", CONFIG['tracking_state']),
], columns=["Parameter", "Value"]).set_index("Parameter")

print("📊 Experiment Configuration:")
display(config_df)

---

<a id="3-environment-setup"></a>
## 3️⃣ Environment Setup

Create the **F-16 longitudinal flight dynamics** environment with a step reference signal for angle-of-attack tracking.

In [ ]:
# Generate time array
tp = generate_time_period(tn=CONFIG["tn"], dt=CONFIG["dt"])
tps = convert_tp_to_sec_tp(tp, dt=CONFIG["dt"])
number_time_steps = len(tp)

# Create step reference signal (radians) using **seconds** timebase
reference_signals = np.reshape(
    unit_step(
        tp=np.asarray(tps, dtype=np.float32),
        degree=CONFIG["step_amplitude_deg"],
        time_step=CONFIG["step_time"],
        output_rad=True,
    ),
    (1, -1),
)

print(f"📈 Reference signal shape: {reference_signals.shape}")
print(f"⏱️  Total time steps: {number_time_steps}")
print(f"🟦 Step occurs at t = {CONFIG['step_time']} s")

In [ ]:
# Create the F-16 longitudinal environment
env = gym.make(
    'LinearLongitudinalF16-v0',
    number_time_steps=number_time_steps, 
    initial_state=[[0], [0], [0], [0]],
    reference_signal=reference_signals,
    tracking_states=[CONFIG["tracking_state"]]
)
env.reset()

print(f"✅ Environment '{env.spec.id}' created successfully!")
print(f"🎯 Tracking state: {CONFIG['tracking_state']} (angle of attack)")

### 📊 System State-Space Matrix (A)

The linearized F-16 model uses state-space representation. Let's examine the A-matrix:

In [ ]:
# Display the discrete-time A-matrix
A_matrix = pd.DataFrame(
    data=env.unwrapped.model.filt_A, 
    columns=env.unwrapped.model.selected_states,
    index=env.unwrapped.model.selected_states
)
print("🔢 Discrete-time State Matrix (A):")
display(A_matrix.round(6))

---

<a id="4-ihdp-agent-configuration"></a>
## 4️⃣ IHDP Agent Configuration

The IHDP agent consists of three main components:

| Component | Purpose |
|-----------|--------|
| **Actor** | Neural network that generates control actions |
| **Critic** | Evaluates the quality of state-action pairs |
| **Incremental** | Handles input constraints and rate limiting |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#                            ACTOR NETWORK SETTINGS
# ═══════════════════════════════════════════════════════════════════════════════
actor_settings = {
    "start_training": 5,                    # Delay before training starts
    "layers": (25, 1),                      # Network architecture
    "activations": ('tanh', 'tanh'),        # Activation functions
    "learning_rate": 2.0 / 25.0,
    "learning_rate_min": 0.001 / 25.0,                     # Learning rate
    "learning_rate_exponent_limit": 10,     # LR exponent limit
    "type_PE": "combined",                  # Persistent excitation type
    "amplitude_3211": 15,                   # 3-2-1-1 signal amplitude
    "pulse_length_3211": 5 / CONFIG["dt"], # Pulse length
    "maximum_input": 25,                    # Max input constraint [deg]
    "maximum_q_rate": 20,                   # Max pitch rate [deg/s]
    "WB_limits": 30,                        # Weight/bias limits
    "NN_initial": 120,                      # Initial NN scaling
    "cascade_actor": False,                 # Cascaded actor structure
    "learning_rate_cascaded": 1.2           # Cascaded LR
}

print("🎭 Actor settings configured")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#                           CRITIC NETWORK SETTINGS
# ═══════════════════════════════════════════════════════════════════════════════
critic_settings = {
    "Q_weights": [8],                       # Q-function weights
    "start_training": -1,                   # Start training immediately
    "gamma": 0.99,                          # Discount factor
    "learning_rate": 15,                    # Learning rate
    "learning_rate_exponent_limit": 10,     # LR exponent limit
    "layers": (25, 1),                      # Network architecture
    "activations": ("tanh", "linear"),      # Activation functions
    "WB_limits": 30,                        # Weight/bias limits
    "NN_initial": 120,                      # Initial NN scaling
    "indices_tracking_states": env.unwrapped.indices_tracking_states
}

print("🧠 Critic settings configured")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#                         INCREMENTAL CONTROL SETTINGS
# ═══════════════════════════════════════════════════════════════════════════════
incremental_settings = {
    "number_time_steps": number_time_steps,
    "dt": CONFIG["dt"],
    "input_magnitude_limits": 25,           # Max elevator deflection [deg]
    "input_rate_limits": 60,                # Max deflection rate [deg/s]
}

print("📐 Incremental settings configured")

In [ ]:
# Create the IHDP agent
agent = IHDPAgent(
    actor_settings, 
    critic_settings, 
    incremental_settings, 
    env.unwrapped.tracking_states, 
    env.unwrapped.state_space, 
    env.unwrapped.control_space, 
    number_time_steps, 
    env.unwrapped.indices_tracking_states
)

print("✅ IHDP Agent created successfully!")

---

<a id="5-simulation-with-failure-injection"></a>
## 5️⃣ Simulation with Failure Injection

### 🚨 Failure Scenario

At step **2500** (t = 25s), we modify the **A-matrix element `A[1][1]`** to simulate a sudden change in aircraft dynamics — representing scenarios like:

- Structural damage
- Control surface failure
- Aerodynamic coefficient change

> ⚠️ **Watch how IHDP adapts!** The neural network controller should compensate for the changed dynamics.

In [ ]:
# Initialize state
xt = np.array([[0], [0]])

# Store original A-matrix value for comparison
original_A11 = env.unwrapped.model.filt_A[1][1]

print(f"🛫 Starting simulation...")
print(f"   Original A[1][1] = {original_A11:.6f}")
print(f"   Failure will occur at step {CONFIG['failure_step']} (t = {CONFIG['failure_step'] * CONFIG['dt']:.1f}s)")
print()

# Main simulation loop
for step in tqdm(range(number_time_steps - 1), desc="🔄 Running simulation"):
    
    # ═══════════════════════════════════════════════════════════════════════════
    #                          FAILURE INJECTION POINT
    # ═══════════════════════════════════════════════════════════════════════════
    if step == CONFIG["failure_step"]:
        print(f"\n🚨 FAILURE INJECTED at step {step}!")
        print(f"   Before: A[1][1] = {env.unwrapped.model.filt_A[1][1]:.6f}")
        env.unwrapped.model.filt_A[1][1] = CONFIG["failure_value"]
        print(f"   After:  A[1][1] = {env.unwrapped.model.filt_A[1][1]:.6f}")
    
    # Get control action from IHDP agent
    ut = agent.predict(xt, reference_signals, step)
    
    # Step the environment
    xt, reward, terminated, truncated, info = env.step(np.array(ut))
    done = terminated or truncated

print("\n✅ Simulation complete!")

---

<a id="6-results-visualization"></a>
## 6️⃣ Results Visualization

### 📈 Angle of Attack (α) Tracking

Compare the reference signal with the actual response. Notice the behavior change after failure injection at t = 25s.

In [ ]:
# Plot angle of attack: reference vs actual
env.unwrapped.model.plot_transient_process(
    'alpha', 
    tps, 
    reference_signals[0], 
    to_deg=True, 
    figsize=(15, 5)
)

print(f"📍 Failure injection point: t = {CONFIG['failure_step'] * CONFIG['dt']:.1f}s")

### 📉 Pitch Rate (q) Response

The pitch rate shows how the aircraft rotates around its lateral axis.

In [ ]:
# Plot pitch rate
env.unwrapped.model.plot_state(
    'wz', 
    tps, 
    to_deg=True, 
    figsize=(15, 5)
)

### 🎮 Control Input (Elevator Deflection)

Observe how the IHDP controller adjusts the elevator command after the failure.

In [ ]:
# Plot elevator deflection
env.unwrapped.model.plot_control(
    'ele', 
    tps, 
    figsize=(15, 5)
)

---

<a id="summary"></a>
## 📝 Summary

### Key Observations

| Phase | Time | Behavior |
|-------|------|----------|
| **Normal operation** | 0 - 25s | Controller tracks the step reference |
| **Failure injection** | t = 25s | A-matrix element modified |
| **Adaptation phase** | 25 - 40s | IHDP compensates for changed dynamics |

### What We Demonstrated

✅ **IHDP's adaptive capability** — The neural network controller adjusts its behavior when system dynamics change unexpectedly.

✅ **Real-time compensation** — No retuning or retraining required; adaptation happens online during the episode.

✅ **Practical fault tolerance** — This approach is relevant for safety-critical aerospace applications.

---

## 🚀 Next Steps

| Action | Description |
|--------|-------------|
| **Experiment with failure parameters** | Try different `failure_step` and `failure_value` settings |
| **Add benchmark metrics** | Import `ControlBenchmark` to quantify performance |
| **Compare controllers** | Run the same scenario with PID or MPC controllers |
| **Multiple failures** | Inject several failures at different times |

```python
# Example: Using ControlBenchmark for metrics
from tensoraerospace.benchmark import ControlBenchmark

benchmark = ControlBenchmark()
metrics = benchmark.plot(
    np.rad2deg(reference_signals[0]),
    env.unwrapped.model.state_history['alpha'],
    dt=CONFIG['dt'],
    tps=tps,
    title="IHDP F-16 Failure Response"
)
```

---

<div align="center">

**📚 Documentation:** [TensorAeroSpace Docs](https://github.com/TensorAeroSpace/TensorAeroSpace)

**🐛 Issues:** [Report a bug](https://github.com/TensorAeroSpace/TensorAeroSpace/issues)

**⭐ Star us on GitHub if this helped!**

</div>